# MCP Hub — Kaggle GPU Experiment Runner

Add two Kaggle Secrets before running:
- `HUB_URL`: Your Oracle Cloud HTTPS URL (e.g. `https://mcp.yourdomain.com`)
- `WORKER_TOKEN`: Your `WORKER_TOKEN` from `.env`

This notebook claims the next queued `kaggle` experiment (or pulls a specific `EXP_ID`), resolves linked Kaggle datasets under `/kaggle/input/`, runs PyTorch GPU training, and posts metrics back to Oracle Cloud (which also triggers your Telegram/Webhook notification).

In [ ]:
import json
import os
from pathlib import Path
import urllib.request
import torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HUB_URL = secrets.get_secret('HUB_URL').rstrip('/')
WORKER_TOKEN = secrets.get_secret('WORKER_TOKEN')
EXP_ID = os.environ.get('EXP_ID', '')  # Leave blank to auto-claim next queued job

def hub_request(method: str, path: str, body: dict | None = None) -> dict:
    data = json.dumps(body).encode('utf-8') if body is not None else None
    req = urllib.request.Request(
        f'{HUB_URL}{path}',
        data=data,
        method=method,
        headers={
            'Authorization': f'Bearer {WORKER_TOKEN}',
            'Content-Type': 'application/json',
        },
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode('utf-8'))

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
hub_request('POST', '/api/nodes/heartbeat', {'name': 'kaggle', 'kind': 'kaggle', 'status': 'online', 'capabilities': {'gpu': gpu_name}})
print('Connected to Hub | Device:', gpu_name)

In [ ]:
if EXP_ID:
    job = hub_request('GET', f'/api/jobs/{EXP_ID}')
else:
    job = hub_request('POST', '/api/jobs/claim', {'node': 'kaggle'}).get('job')

assert job is not None, 'No queued Kaggle experiment found in Hub.'
exp_id = job['id']
config = job.get('config', {})
dataset_path = config.get('dataset_path') or '/kaggle/input'
print(f'Running {exp_id}: {job["name"]} | Dataset slug: {job.get("dataset_slug", "-")}')
print('Expected dataset path:', dataset_path)
if Path('/kaggle/input').exists():
    print('Mounted Kaggle datasets:', [p.name for p in Path('/kaggle/input').iterdir()])
print('Config:', json.dumps(config, indent=2))

In [ ]:
# Replace this block with your real PyTorch training & evaluation pipeline reading from `dataset_path`
epochs = int(config.get('epochs', 5))
lr = float(config.get('lr', 1e-3))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

logs = [f'Starting training on {device} | dataset={dataset_path} | epochs={epochs}, lr={lr}']
metrics = {
    'accuracy': 0.912,
    'f1': 0.905,
    'precision': 0.908,
    'recall': 0.902,
    'loss': 0.214,
    'device': device,
}
logs.append(f'Finished training | metrics={metrics}')
print('\n'.join(logs))

In [ ]:
result = hub_request(
    'POST',
    f'/api/jobs/{exp_id}/complete',
    {
        'status': 'completed',
        'metrics': metrics,
        'logs': '\n'.join(logs),
        'artifact_url': '',
    },
)
print('Reported result to Hub:', json.dumps(result, indent=2))